# SubArabify — Colab T4 transcription backend (v1.0.4-pre)

Turns a **free Google Colab T4 GPU** into an Arabic-subtitle factory for full movies.

* Model: [`samil24/whisper-large-arabic-dialects-v5`](https://huggingface.co/samil24/whisper-large-arabic-dialects-v5)
  (openai/whisper-large-v3 fine-tuned on Arabic dialects — direct Arabic transcription, no EN→AR step)
* Exposed through a local **FastAPI** server + a **free public tunnel**
  (Cloudflare quick tunnel — no signup; pyngrok used as fallback if `NGROK_AUTHTOKEN` is set)
* The machine you run `colab/client.py` on only does ffmpeg extraction + upload —
  all heavy lifting happens here on the T4

> **Why `transformers` and not `faster-whisper`?** This HF repo is a raw Transformers
> checkpoint (safetensors, 2B params). faster-whisper only loads converted CTranslate2
> models, so we load it exactly the way the model card shows.

## How to use
1. **Runtime → Run all** (make sure the runtime type is **GPU / T4** — the first cell checks)
2. Wait for the last cell to print `PUBLIC URL: https://….trycloudflare.com` and `✔ tunnel OK`
3. On your machine:
   ```bash
   python colab/client.py "My Movie.mkv" --url https://xxxx.trycloudflare.com
   ```
4. Keep this tab open while jobs run — Colab disconnects idle sessions, and every new
   session gets a **new** URL (the client prints a clear error if the URL went stale).

A 2-hour movie takes roughly **15–20 minutes** on a T4, which is why the API is
**asynchronous** (submit job → poll progress → download `.srt`): a single HTTP request
that long would die on any network hiccup. If your connection drops, resume the same
job later with `--job <id>`.

In [ ]:
# @title 1 · Environment: GPU, packages, tunnel binary
import os, sys, urllib.request

!nvidia-smi -L

# torch (CUDA build) ships with Colab — we only add what we need on top
%pip install -q -U "transformers>=4.45" "accelerate>=0.34" "huggingface_hub>=0.24" \
    "fastapi>=0.111" "uvicorn[standard]" python-multipart pyngrok

# cloudflared = free public tunnel, no account needed
if not os.path.exists("/usr/local/bin/cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "/usr/local/bin/cloudflared",
    )
    os.chmod("/usr/local/bin/cloudflared", 0o755)
    print("cloudflared installed")

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), "Runtime → Change runtime type → GPU (T4)"

In [ ]:
# @title 2 · Download the Arabic Whisper model (once per session)
from huggingface_hub import snapshot_download

MODEL_ID = "samil24/whisper-large-arabic-dialects-v5"
MODEL_PATH = snapshot_download(MODEL_ID)          # ~fine-tuned whisper-large-v3, cached
print("model cached at", MODEL_PATH)

In [ ]:
# @title 3 · FastAPI backend: async jobs + Arabic transcription + SRT
"""SubArabify Colab backend v1.0.4-pre.

Model: samil24/whisper-large-arabic-dialects-v5 — openai/whisper-large-v3
fine-tuned on Arabic dialects. Loaded with `transformers` (fp16 on the T4)
exactly as the model card prescribes; faster-whisper cannot load raw HF
checkpoints without a CTranslate2 conversion, so it is not used here.

API (async by design — a 2 h movie takes ~15-20 min on a T4):
  GET    /health            liveness + model state
  POST   /jobs              multipart upload -> {"job_id": ...}        (202)
  POST   /jobs/url          {"url": "..."} direct audio link -> job    (202)
  GET    /jobs/{id}         poll {"status", "progress", "message"}
  GET    /jobs/{id}/srt     finished .srt text (409 until done)
  POST   /transcribe        sync upload -> .srt text (short clips only)
"""
import queue, re, shutil, subprocess, tempfile, threading, time, uuid, wave
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import requests as rq
import torch
from fastapi import FastAPI, File, HTTPException, UploadFile
from fastapi.responses import PlainTextResponse
from pydantic import BaseModel

APP_VERSION = "1.0.4-pre"
MODEL_ID = "samil24/whisper-large-arabic-dialects-v5"
SR = 16000                 # Whisper's native sample rate
WIN_S = 300.0              # transcribe 5 min of audio per progress step
HOP_S = 290.0              # 10 s overlap between steps (midpoint-stitched)
MAX_BYTES = 2_500_000_000  # 2.5 GB upload/download cap
GEN = {"language": "ar", "task": "transcribe"}


# ── audio ────────────────────────────────────────────────────────────────

def find_ffmpeg():
    for c in ("/usr/bin/ffmpeg", shutil.which("ffmpeg") or "",
              "/usr/lib/jellyfin-ffmpeg/ffmpeg"):
        if c and Path(c).exists():
            return c
    raise RuntimeError("ffmpeg not found (Colab ships one; else apt install ffmpeg)")


def _ffmpeg(args, timeout=7200):
    r = subprocess.run([find_ffmpeg(), "-nostdin", "-y", "-v", "error"] + args,
                       capture_output=True, text=True, timeout=timeout)
    if r.returncode != 0:
        raise RuntimeError("ffmpeg: " + r.stderr.strip()[:400])


def to_wav16k(src: Path) -> Path:
    """Any audio/video container -> 16 kHz mono PCM WAV (Whisper's input)."""
    out_dir = Path(tempfile.mkdtemp(prefix="sbc-audio-"))
    wav = out_dir / "audio.wav"
    _ffmpeg(["-i", str(src), "-map", "0:a:0", "-ar", "16000", "-ac", "1",
             "-c:a", "pcm_s16le", str(wav)])
    if not wav.exists() or wav.stat().st_size <= 44:
        raise RuntimeError("no decodable audio track in the upload")
    return wav


def load_wav(path: Path) -> np.ndarray:
    with wave.open(str(path), "rb") as w:
        if w.getframerate() != SR or w.getsampwidth() != 2:
            raise RuntimeError("unexpected wav layout")
        data = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16)
    return data.astype(np.float32) / 32768.0


# ── model (T4 fp16, lazy-loaded) ────────────────────────────────────────

_PIPE = None
_PIPE_LOCK = threading.Lock()


def get_pipe():
    global _PIPE
    with _PIPE_LOCK:
        if _PIPE is None:
            from transformers import pipeline as hf_pipeline
            if not torch.cuda.is_available():
                raise RuntimeError("no GPU — Runtime → Change runtime type → T4 GPU")
            t0 = time.time()
            _PIPE = hf_pipeline("automatic-speech-recognition", model=MODEL_ID,
                                torch_dtype=torch.float16, device=0)
            print(f"[model] ready on {torch.cuda.get_device_name(0)} in {time.time()-t0:.0f}s")
        return _PIPE


# ── transcription: overlapping windows with real progress ───────────────

def _stitch(windows):
    """Overlapping windows -> one timeline. Each segment is owned by exactly
    one window: the midpoint of every overlap region is the cut line, so no
    word is dropped or duplicated at a seam."""
    n = len(windows)
    bounds = [(windows[i + 1][0] + windows[i][0] + windows[i][1]) / 2
              for i in range(n - 1)]
    out = []
    for j, (_s, _d, segs) in enumerate(windows):
        for seg in segs:
            mid = (seg["s"] + seg["e"]) / 2
            if j > 0 and mid < bounds[j - 1]:
                continue
            if j < n - 1 and mid >= bounds[j]:
                continue
            out.append(seg)
    out.sort(key=lambda x: x["s"])
    return out


def transcribe_wav(wav_path: Path, on_progress):
    audio = load_wav(wav_path)
    total = len(audio)
    if total < SR // 10:
        raise RuntimeError("audio is shorter than 0.1 s")
    pipe = get_pipe()
    windows, start_s = [], 0.0
    while True:
        s0 = int(start_s * SR)
        s1 = min(total, int((start_s + WIN_S) * SR))
        seg = audio[s0:s1]
        if seg.size == 0:
            break
        out = pipe(seg, chunk_length_s=30, stride_length_s=5,
                   return_timestamps=True, generate_kwargs=dict(GEN))
        w_start, w_dur = s0 / SR, seg.size / SR
        segs = []
        for ch in out.get("chunks") or ():
            ts = ch.get("timestamp") or (None, None)
            a = w_start + (0.0 if ts[0] is None else float(ts[0]))
            b = w_start + (w_dur if ts[1] is None else float(ts[1]))
            txt = re.sub(r"\s+", " ", ch.get("text") or "").strip()
            if txt and b > a:
                segs.append({"s": a, "e": b, "text": txt})
        windows.append((w_start, w_dur, segs))
        done = s1 >= total
        on_progress(min(0.985, s1 / total),
                    f"window {len(windows)}{' (last)' if done else ''} · "
                    f"{int(s1 // SR)}s / {int(total // SR)}s")
        if done:
            break
        start_s += HOP_S
    return _stitch(windows)


# ── segments -> plain Arabic .srt (branding stays in the client) ────────

def _pack(text, limit=84):
    """Split a long segment into cue-sized pieces on sentence then word edges."""
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    if len(text) <= limit:
        return [text]
    sents = [s.strip() for s in re.findall(r"[^.!?…?]+[.!?…?]*", text)
             if s.strip()] or [text]
    chunks, cur = [], ""
    for s in sents:
        if not cur:
            cur = s
        elif len(cur) + 1 + len(s) <= limit:
            cur = cur + " " + s
        else:
            chunks.append(cur)
            cur = s
    if cur:
        chunks.append(cur)
    out = []
    for c in chunks:
        if len(c) <= limit:
            out.append(c)
            continue
        words, cur = c.split(), ""
        for w in words:
            if cur and len(cur) + 1 + len(w) > limit:
                out.append(cur)
                cur = w
            else:
                cur = (cur + " " + w).strip()
        if cur:
            out.append(cur)
    return out


def _tc(t):
    ms = max(0, int(round(t * 1000)))
    h, rem = divmod(ms, 3600000)
    m, rem = divmod(rem, 60000)
    s, ms = divmod(rem, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def segs_to_srt(segs) -> str:
    """Plain (unbranded) SRT. The client runs the result through srtcore, so
    app / Jellyfin addon / Colab outputs share identical branding rules."""
    lines, prev_end, idx = [], 0.0, 1
    for seg in segs:
        parts = _pack(seg["text"]) or [seg["text"]]
        span = max(seg["e"] - seg["s"], 0.2)
        chars = sum(len(p) for p in parts) or 1
        t = max(seg["s"], prev_end)
        for k, p in enumerate(parts):
            e = seg["e"] if k == len(parts) - 1 else t + span * len(p) / chars
            e = max(e, t + 0.05)
            lines += [str(idx), f"{_tc(t)} --> {_tc(e)}", p, ""]
            idx += 1
            t = e
            prev_end = e
    if idx == 1:
        raise RuntimeError("model heard no speech (silent / music-only audio?)")
    return "\n".join(lines)


# ── job manager (single worker: one T4, one movie at a time) ────────────

class Job:
    __slots__ = ("id", "status", "progress", "message", "srt",
                 "created", "source", "input_path")

    def __init__(self, source, input_path=None):
        self.id = uuid.uuid4().hex[:12]
        self.status = "queued"        # queued -> processing -> done | error
        self.progress = 0.0
        self.message = "queued"
        self.srt = None
        self.created = time.time()
        self.source = source
        self.input_path = input_path

    def as_dict(self):
        return {"job_id": self.id, "status": self.status,
                "progress": round(self.progress, 3), "message": self.message,
                "source": self.source,
                "elapsed_s": round(time.time() - self.created, 1)}


JOBS, JOBS_LOCK = {}, threading.Lock()
JOB_Q = queue.Queue()


def _cleanup(job):
    if job.input_path:
        shutil.rmtree(Path(job.input_path).parent, ignore_errors=True)


def put_job(job):
    with JOBS_LOCK:
        finished = sorted((j for j in JOBS.values() if j.status in ("done", "error")),
                          key=lambda j: j.created)
        while len(JOBS) >= 8 and finished:
            old = finished.pop(0)
            JOBS.pop(old.id, None)
            _cleanup(old)
        JOBS[job.id] = job
    JOB_Q.put(job.id)


def get_job(jid):
    with JOBS_LOCK:
        return JOBS.get(jid)


def run_job(job):
    wav_dir = None
    try:
        with JOBS_LOCK:
            job.status, job.message, job.progress = "processing", "decoding audio", 0.01
        wav = to_wav16k(job.input_path)
        wav_dir = wav.parent

        def cb(p, note):
            with JOBS_LOCK:
                job.progress = 0.02 + 0.96 * p
                job.message = note

        segs = transcribe_wav(wav, cb)
        srt = segs_to_srt(segs)
        with JOBS_LOCK:
            job.srt, job.status, job.progress = srt, "done", 1.0
            job.message = f"{len(segs)} cues"
        print(f"[job {job.id}] done: {len(segs)} cues from {job.source}")
    except Exception as e:   # surface everything to the polling client
        with JOBS_LOCK:
            job.status = "error"
            job.message = (f"{type(e).__name__}: {e}")[:500]
        print(f"[job {job.id}] ERROR {job.message}")
    finally:
        if wav_dir:
            shutil.rmtree(wav_dir, ignore_errors=True)
        _cleanup(job)


def _worker():
    while True:
        job = get_job(JOB_Q.get())
        if job is not None:
            run_job(job)


threading.Thread(target=_worker, daemon=True, name="sb-worker").start()


# ── HTTP API ─────────────────────────────────────────────────────────────

app = FastAPI(title="SubArabify Colab backend", version=APP_VERSION)


class UrlIn(BaseModel):
    url: str


async def _save_upload(up: UploadFile, dest: Path):
    n = 0
    with open(dest, "wb") as f:
        while True:
            chunk = await up.read(1 << 20)
            if not chunk:
                break
            n += len(chunk)
            if n > MAX_BYTES:
                raise HTTPException(413, "file too large (2.5 GB max)")
            f.write(chunk)
    if n == 0:
        raise HTTPException(400, "empty upload")
    return n


def _job_dir(filename):
    ext = Path(filename or "audio.bin").suffix[:12] or ".bin"
    return Path(tempfile.mkdtemp(prefix="sb-job-")) / f"input{ext}"


@app.get("/")
def root():
    return {"service": "subarabify-colab", "version": APP_VERSION, "model": MODEL_ID,
            "endpoints": ["GET /health", "POST /jobs (multipart file)",
                          'POST /jobs/url ({"url": ...})', "GET /jobs/{id}",
                          "GET /jobs/{id}/srt", "POST /transcribe (sync)"]}


@app.get("/health")
def health():
    return {"status": "ok", "version": APP_VERSION, "model": MODEL_ID,
            "model_loaded": _PIPE is not None,
            "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "queued": JOB_Q.qsize()}


@app.post("/jobs", status_code=202)
async def create_job(file: UploadFile = File(...)):
    dest = _job_dir(file.filename)
    await _save_upload(file, dest)
    job = Job(source=Path(file.filename or "upload").name, input_path=dest)
    put_job(job)
    d = job.as_dict()
    d.update({"status_url": f"/jobs/{job.id}", "srt_url": f"/jobs/{job.id}/srt"})
    return d


@app.post("/jobs/url", status_code=202)
def create_job_url(body: UrlIn):
    url = body.url.strip()
    if not re.match(r"^https?://", url):
        raise HTTPException(400, "url must be http(s)://")
    dest = _job_dir(Path(urlparse(url).path).name or "remote.bin")
    try:
        with rq.get(url, stream=True, timeout=(15, 600),
                    headers={"User-Agent": f"SubArabify/{APP_VERSION}"}) as r:
            r.raise_for_status()
            n = 0
            with open(dest, "wb") as f:
                for chunk in r.iter_content(1 << 20):
                    n += len(chunk)
                    if n > MAX_BYTES:
                        raise HTTPException(413, "remote file too large (2.5 GB max)")
                    f.write(chunk)
        if n == 0:
            raise HTTPException(400, "remote file is empty")
    except HTTPException:
        shutil.rmtree(dest.parent, ignore_errors=True)
        raise
    except Exception as e:
        shutil.rmtree(dest.parent, ignore_errors=True)
        raise HTTPException(400, f"download failed: {e}")
    job = Job(source=url, input_path=dest)
    put_job(job)
    d = job.as_dict()
    d.update({"status_url": f"/jobs/{job.id}", "srt_url": f"/jobs/{job.id}/srt"})
    return d


@app.get("/jobs/{jid}")
def job_status(jid: str):
    job = get_job(jid)
    if job is None:
        raise HTTPException(404, "unknown job")
    return job.as_dict()


@app.get("/jobs/{jid}/srt", response_class=PlainTextResponse)
def job_srt(jid: str):
    job = get_job(jid)
    if job is None:
        raise HTTPException(404, "unknown job")
    if job.status != "done" or job.srt is None:
        raise HTTPException(409, f"job is {job.status}: {job.message}")
    return job.srt


@app.post("/transcribe", response_class=PlainTextResponse)
def transcribe_sync(file: UploadFile = File(...)):
    """Blocking upload -> .srt text. Fine for short clips; movies use /jobs
    (defined as a plain def, so FastAPI runs it in its thread pool and the
    event loop keeps answering /health while it thinks)."""
    dest = _job_dir(file.filename)
    import anyio
    # UploadFile.read must be awaited; bridge it for the sync endpoint
    async def _drain():
        await _save_upload(file, dest)
    anyio.run(_drain)
    job = Job(source=Path(file.filename or "upload").name, input_path=dest)
    run_job(job)
    if job.status == "error":
        raise HTTPException(500, job.message)
    return job.srt

In [ ]:
# @title 4 · Start the server + warm the model
import threading, time

import requests
import uvicorn

server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000,
                                       log_level="info", access_log=False))
threading.Thread(target=server.run, daemon=True, name="sb-uvicorn").start()

for _ in range(60):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.ok:
            print("server up:", r.json())
            break
    except Exception:
        time.sleep(0.5)
else:
    raise RuntimeError("server did not start — check the cell above")

# load the model now so the first real job doesn't wait for it
threading.Thread(target=get_pipe, daemon=True).start()
print("model warming in background…")

In [ ]:
# @title 5 · Open the public tunnel + print the client command
import os, queue, re, subprocess, threading, time
from pathlib import Path

def start_cloudflared(port=8000):
    """Free Cloudflare quick tunnel (no signup). Returns (proc, url)."""
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}",
         "--no-autoupdate"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1024)
    q = queue.Queue()
    def pump():
        for line in proc.stdout:
            q.put(line)
    threading.Thread(target=pump, daemon=True).start()
    deadline, logs = time.time() + 60, []
    while time.time() < deadline:
        try:
            line = q.get(timeout=0.5)
        except queue.Empty:
            if proc.poll() is not None:
                break
            continue
        logs.append(line.rstrip())
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if m:
            return proc, m.group(0), logs
    return proc, None, logs

def start_ngrok(port=8000):
    """Fallback: pyngrok, if the user provided NGROK_AUTHTOKEN (secret)."""
    token = os.environ.get("NGROK_AUTHTOKEN")
    if not token:
        return None, None
    from pyngrok import conf, ngrok
    conf.get_default().auth_token = token
    tunnel = ngrok.connect(port, "http")
    return tunnel, tunnel.public_url

tunnel, PUBLIC_URL = None, None
cf_proc, url, logs = start_cloudflared()
if url:
    tunnel = cf_proc          # stop() below handles Popen
    PUBLIC_URL = url
else:
    print("cloudflared did not print a URL; logs:")
    print("\n".join(logs[-30:]))
    print("→ trying pyngrok…")
    tunnel, PUBLIC_URL = start_ngrok()

if not PUBLIC_URL:
    raise RuntimeError("no tunnel could be opened — see logs above "
                       "(set NGROK_AUTHTOKEN to use ngrok instead)")

PUBLIC_URL = PUBLIC_URL.rstrip("/")
Path("/content/subarabify_url.txt").write_text(PUBLIC_URL)

# end-to-end check through the public URL (first request can be slow)
r = requests.get(PUBLIC_URL + "/health", timeout=60)
r.raise_for_status()

print("=" * 70)
print("PUBLIC URL:", PUBLIC_URL)
print("✔ tunnel OK —", r.json())
print()
print("Client command (from your SubArabify checkout):")
print(f'  python colab/client.py "My Movie.mkv" --url {PUBLIC_URL}')
print()
print("Tip: %cat /content/subarabify_url.txt re-reads this URL any time.")
print("Keep this Colab tab open while jobs run.")
print("=" * 70)

## API at a glance

| Endpoint | What it does |
|---|---|
| `GET /health` | liveness, model state, queue depth |
| `POST /jobs` | multipart upload (`file=…`) → `202 {"job_id": …}` |
| `POST /jobs/url` | `{"url": "https://…/audio.wav"}` → server downloads it → job |
| `GET /jobs/{id}` | `{"status": "queued\|processing\|done\|error", "progress": 0–1, "message": …}` |
| `GET /jobs/{id}/srt` | the finished `.srt` text (`409` while not done) |
| `POST /transcribe` | blocking upload → `.srt` — **short clips only**, movies use `/jobs` |

```bash
# quick sync test with a short clip
curl -F "file=@clip.wav"  "$PUBLIC_URL/transcribe"

# the real path (what colab/client.py does)
curl -F "file=@movie.wav" "$PUBLIC_URL/jobs"                 # -> job_id
curl "$PUBLIC_URL/jobs/<job_id>"                              # poll
curl "$PUBLIC_URL/jobs/<job_id>/srt"                          # fetch
```

## Notes & troubleshooting

* **~15–20 min for a 2-hour movie on a T4** — that's why jobs are async. The client
  polls every 15 s and tolerates transient tunnel blips; if the whole connection dies,
  resume with `python colab/client.py --url $URL --job <job_id> --save-to "Movie.mkv"`.
* **URL changes every session** — Colab quick-tunnel URLs are throwaway. After a
  disconnect, re-copy the URL from this notebook.
* **Colab sleeps/disconnects** — keep the tab open; free sessions also cap total runtime.
  A dropped session kills in-flight jobs (client fails with a clear message; re-submit).
* **Uploads > ~90 MB** — Cloudflare quick tunnels cap request bodies near 100 MB, so the
  client auto-switches the 16 kHz WAV to Opus (`--transport wav` forces raw).
* **ngrok instead of cloudflared** — set the Colab secret `NGROK_AUTHTOKEN` and re-run
  the tunnel cell; the client works with either URL.
* **Batch** — one job at a time by design (single T4); extra submissions queue.

In [ ]:
# @title (optional) Stop server + tunnel
try:
    server.should_exit = True
except NameError:
    pass

t = globals().get("tunnel")
if t is not None:
    try:
        if hasattr(t, "terminate"):      # cloudflared Popen
            t.terminate()
        else:                            # pyngrok tunnel object
            import pyngrok.ngrok as _ng
            _ng.disconnect(t.public_url)
    except Exception as e:
        print("tunnel stop:", e)

print("stopped. Re-run cell 5 to open a fresh URL (model + server stay up).")